In [1]:
import pandas as pd  
import numpy as np
from xgboost import XGBClassifier

from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [3]:
df = pd.read_csv('../Data/team_stats3.csv')

In [4]:
# Load valid team names
with open('../Data/Dictionaries/team_names.txt', encoding='utf-8') as f:
    valid_teams = set(line.strip() for line in f if line.strip())

# Filter DataFrame to only include rows where both team and opponent are valid
df = df[df['team'].isin(valid_teams) & df['opponent'].isin(valid_teams)]

# Remove rows with missing kills, deaths, or assists
df = df.dropna(subset=['total_kills','total_deaths','total_assists','avg_acs'])

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

In [5]:
df.head()

,patch,date,map,team,opponent,won_map,won_series,series_score,rounds_played,rounds_won,...,avg_acs_defense,avg_kast,total_first_kills,total_first_deaths,agent1,agent2,agent3,agent4,agent5,map_url
18463,NaN,2020-09-23,Split,Rex Regum Qeon,BOOM Esports,False,True,2:0,20,7,...,63.6,0,6,14,sage,raze,breach,omen,cypher,https://www.vlr.gg/2902/rrq-endeavour-vs-boom-...
18464,NaN,2020-09-23,Split,BOOM Esports,Rex Regum Qeon,True,False,2:0,20,13,...,152.8,0,14,6,jett,omen,raze,breach,cypher,https://www.vlr.gg/2902/rrq-endeavour-vs-boom-...
18465,NaN,2020-09-23,Haven,Rex Regum Qeon,BOOM Esports,False,True,2:0,16,3,...,131.0,0,5,11,sova,raze,omen,cypher,breach,https://www.vlr.gg/2902/rrq-endeavour-vs-boom-...
18466,NaN,2020-09-23,Haven,BOOM Esports,Rex Regum Qeon,True,False,2:0,16,13,...,62.2,0,11,5,breach,reyna,jett,omen,sova,https://www.vlr.gg/2902/rrq-endeavour-vs-boom-...
20389,NaN,2020-10-28,Ascent,BBL Esports,FUT Esports,False,True,2:0,19,6,...,69.8,0,8,11,jett,phoenix,killjoy,sova,omen,https://www.vlr.gg/4389/bbl-esports-vs-futboli...


Calculate more stats and add to dataframe

In [6]:
def weighted_average(series):
    weights = np.linspace(1, 2, len(series))  # More weight to recent
    return np.average(series, weights=weights) if len(series) > 0 else 0

all_data_stats = []
for idx, row in df.iterrows():
    match_date = row['date']
    team = row['team']
    opponent = row['opponent']
    map_name = row['map']

    # Team and opponent history up to this match
    team_hist = df[(df['team'] == team) & (df['date'] < match_date)]
    opp_hist = df[(df['team'] == opponent) & (df['date'] < match_date)]

    # --- Recent form (last 5 matches) ---
    team_last_10_games = team_hist.tail(10)
    opp_last_10_games = opp_hist.tail(10)
    team_avg_kills_last10 = team_last_10_games['total_kills'].mean() if not team_last_10_games.empty else 0
    team_weighted_kills_last10 = weighted_average(team_last_10_games['total_kills'])
    team_win_rate_last10 = team_last_10_games['won_map'].mean() if not team_last_10_games.empty else 0
    opp_avg_kills_last10 = opp_last_10_games['total_kills'].mean() if not opp_last_10_games.empty else 0
    opp_win_rate_last10 = opp_last_10_games['won_map'].mean() if not opp_last_10_games.empty else 0

    # --- Head-to-head history ---
    h2h_hist = df[
        (((df['team'] == team) & (df['opponent'] == opponent)) |
         ((df['team'] == opponent) & (df['opponent'] == team)))
        & (df['date'] < match_date)
    ]
    h2h_win_rate = h2h_hist[h2h_hist['team'] == team]['won_map'].mean() if not h2h_hist.empty else 0.5

    # --- Map-specific win rates ---
    team_map_hist = team_hist[team_hist['map'] == map_name]
    opp_map_hist = opp_hist[opp_hist['map'] == map_name]
    team_map_win_rate = team_map_hist['won_map'].mean() if not team_map_hist.empty else 0
    opp_map_win_rate = opp_map_hist['won_map'].mean() if not opp_map_hist.empty else 0

    # --- Other features ---
    team_avg_kills = team_hist['total_kills'].mean() if not team_hist.empty else 0
    team_avg_deaths = team_hist['total_deaths'].mean() if not team_hist.empty else 0
    team_avg_assists = team_hist['total_assists'].mean() if not team_hist.empty else 0
    team_win_rate = team_hist['won_map'].mean() if not team_hist.empty else 0
    opp_avg_kills = opp_hist['total_kills'].mean() if not opp_hist.empty else 0
    opp_win_rate = opp_hist['won_map'].mean() if not opp_hist.empty else 0

    # --- Feature interactions ---
    diff_avg_kills = team_avg_kills - opp_avg_kills
    diff_win_rate = team_win_rate - opp_win_rate
    ratio_win_rate = team_win_rate / opp_win_rate if opp_win_rate > 0 else 0

    # --- Build feature row ---
    all_data_stats.append({
        'date': match_date,
        'team': team,
        'opponent': opponent,
        'map': map_name,
        'team_avg_kills': team_avg_kills,
        'team_win_rate': team_win_rate,
        'team_avg_deaths': team_avg_deaths,
        'team_avg_assists': team_avg_assists,
        'opp_avg_kills': opp_avg_kills,
        'opp_win_rate': opp_win_rate,
        'team_avg_kills_last10': team_avg_kills_last10,
        'team_weighted_kills_last10': team_weighted_kills_last10,
        'team_win_rate_last10': team_win_rate_last10,
        'opp_avg_kills_last10': opp_avg_kills_last10,
        'opp_win_rate_last10': opp_win_rate_last10,
        'h2h_win_rate': h2h_win_rate,
        'team_map_win_rate': team_map_win_rate,
        'opp_map_win_rate': opp_map_win_rate,
        'diff_avg_kills': diff_avg_kills,
        'diff_win_rate': diff_win_rate,
        'ratio_win_rate': ratio_win_rate,
        'won_map': row['won_map']  # label
    })

match_stats = pd.DataFrame(all_data_stats)
match_stats = pd.get_dummies(match_stats, columns=['map'])

In [7]:
match_stats.head()

,date,team,opponent,team_avg_kills,team_win_rate,team_avg_deaths,team_avg_assists,opp_avg_kills,opp_win_rate,team_avg_kills_last10,...,map_Breeze,map_Corrode,map_Fracture,map_Haven,map_Icebox,map_Lotus,map_Pearl,map_Split,map_Sunset,map_TBD
0,2020-09-23,Rex Regum Qeon,BOOM Esports,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,True,False,False
1,2020-09-23,BOOM Esports,Rex Regum Qeon,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,True,False,False
2,2020-09-23,Rex Regum Qeon,BOOM Esports,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,False,False,False,True,False,False,False,False,False,False
3,2020-09-23,BOOM Esports,Rex Regum Qeon,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,False,False,False,True,False,False,False,False,False,False
4,2020-10-28,BBL Esports,FUT Esports,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False


In [8]:
# Removing all the zero rows to avoid issues with model training (optional and maybe make it better or worse))

match_stats = match_stats[(match_stats['team_avg_kills'] > 0) & (match_stats['opp_avg_kills'] > 0)]

In [9]:
match_stats.head()

,date,team,opponent,team_avg_kills,team_win_rate,team_avg_deaths,team_avg_assists,opp_avg_kills,opp_win_rate,team_avg_kills_last10,...,map_Breeze,map_Corrode,map_Fracture,map_Haven,map_Icebox,map_Lotus,map_Pearl,map_Split,map_Sunset,map_TBD
8,2020-12-05,BBL Esports,FUT Esports,61.5,0.0,82.50,19.0,82.50,1.0,61.5,...,False,False,False,False,False,False,False,False,False,False
9,2020-12-05,BBL Esports,FUT Esports,61.5,0.0,82.50,19.0,82.50,1.0,61.5,...,False,False,False,False,False,False,False,False,False,False
10,2020-12-05,FUT Esports,BBL Esports,82.5,1.0,61.50,32.5,61.50,0.0,82.5,...,False,False,False,False,False,False,False,False,False,False
11,2020-12-05,FUT Esports,BBL Esports,82.5,1.0,61.50,32.5,61.50,0.0,82.5,...,False,False,False,False,False,False,False,False,False,False
12,2020-12-06,FUT Esports,BBL Esports,74.0,0.5,70.75,32.0,70.75,0.5,74.0,...,False,False,False,False,False,False,False,False,False,True


In [10]:
# Train/test split (make sure X_train, X_test, y_train, y_test are defined)
# Remove map columns for map-agnostic prediction
feature_cols = [
    'team_avg_kills', 'team_win_rate', 'opp_avg_kills', 'opp_win_rate',
    'team_avg_deaths', 'team_avg_assists',
    'team_avg_kills_last10', 'team_weighted_kills_last10', 'team_win_rate_last10', 'opp_avg_kills_last10', 'opp_win_rate_last10',
    'h2h_win_rate', 'diff_avg_kills', 'diff_win_rate', 'ratio_win_rate'
]

X = match_stats[feature_cols]
y = match_stats['won_map']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42)

# Train the XGBoost classifier
from xgboost import XGBClassifier
clf = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
clf.fit(X_train, y_train)

# Predict and print accuracy
y_pred = clf.predict(X_test)
from sklearn.metrics import accuracy_score
print("Test accuracy:", accuracy_score(y_test, y_pred))

Test accuracy: 0.7012595837897043


/Users/bitanga/Coding/vct-predictions-main/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [21:05:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [11]:
def predict_match_winner(team_a, team_b, as_of_date=None):
    if as_of_date is None:
        as_of_date = df['date'].max()

    # Histories
    team_a_hist = df[(df['team'] == team_a) & (df['date'] < as_of_date)]
    team_b_hist = df[(df['team'] == team_b) & (df['date'] < as_of_date)]

    # Recent form (last 10)
    team_a_avg_kills_last10 = team_a_hist.tail(10)['total_kills'].mean() if not team_a_hist.tail(10).empty else 0
    team_a_weighted_kills_last10 = weighted_average(team_a_hist.tail(10)['total_kills'])
    team_a_win_rate_last10 = team_a_hist.tail(10)['won_map'].mean() if not team_a_hist.tail(10).empty else 0
    team_b_avg_kills_last10 = team_b_hist.tail(10)['total_kills'].mean() if not team_b_hist.tail(10).empty else 0
    team_b_win_rate_last10 = team_b_hist.tail(10)['won_map'].mean() if not team_b_hist.tail(10).empty else 0

    # Head-to-head
    h2h_hist = df[
        (((df['team'] == team_a) & (df['opponent'] == team_b)) |
         ((df['team'] == team_b) & (df['opponent'] == team_a)))
        & (df['date'] < as_of_date)
    ]
    h2h_win_rate = h2h_hist[h2h_hist['team'] == team_a]['won_map'].mean() if not h2h_hist.empty else 0.5

    # Other features
    team_a_avg_kills = team_a_hist['total_kills'].mean() if not team_a_hist.empty else 0
    team_a_avg_deaths = team_a_hist['total_deaths'].mean() if not team_a_hist.empty else 0
    team_a_avg_assists = team_a_hist['total_assists'].mean() if not team_a_hist.empty else 0
    team_a_win_rate = team_a_hist['won_map'].mean() if not team_a_hist.empty else 0
    team_b_avg_kills = team_b_hist['total_kills'].mean() if not team_b_hist.empty else 0
    team_b_win_rate = team_b_hist['won_map'].mean() if not team_b_hist.empty else 0

    # Feature interactions
    diff_avg_kills = team_a_avg_kills - team_b_avg_kills
    diff_win_rate = team_a_win_rate - team_b_win_rate
    ratio_win_rate = team_a_win_rate / team_b_win_rate if team_b_win_rate > 0 else 0

    # Build feature row (no map columns)
    feature_row = {
        'team_avg_kills': team_a_avg_kills,
        'team_win_rate': team_a_win_rate,
        'opp_avg_kills': team_b_avg_kills,
        'opp_win_rate': team_b_win_rate,
        'team_avg_deaths': team_a_avg_deaths,
        'team_avg_assists': team_a_avg_assists,
        'team_avg_kills_last10': team_a_avg_kills_last10,
        'team_weighted_kills_last10': team_a_weighted_kills_last10,
        'team_win_rate_last10': team_a_win_rate_last10,
        'opp_avg_kills_last10': team_b_avg_kills_last10,
        'opp_win_rate_last10': team_b_win_rate_last10,
        'h2h_win_rate': h2h_win_rate,
        'diff_avg_kills': diff_avg_kills,
        'diff_win_rate': diff_win_rate,
        'ratio_win_rate': ratio_win_rate
    }

    feature_row_df = pd.DataFrame([feature_row])

    pred = clf.predict(feature_row_df)[0]
    proba = clf.predict_proba(feature_row_df)[0]
    confidence = proba[int(pred)] * 100

    winner = team_a if pred == 1 else team_b
    print(f"Predicted winner: {winner} (confidence: {confidence:.1f}%)")
    return winner, confidence

In [21]:
predict_match_winner(team_a = "EDG", team_b = "FNATIC")

Predicted winner: FNATIC (confidence: 60.6%)


('FNATIC', np.float32(60.59354))